# VAR-d20 Content and Style VAE Reconstruction

First experiment step: load your content/style images, convert them into VAR/VQ token pyramids, then reconstruct them through the VAE. Do this before changing the autoregressive sampler.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LeeHoang2710/Style-Transfer-Experiment.git"  # GitHub sync source
BRANCH = "main"
WORKSPACE = Path('/content/VAR_Style_Transfer_Workspace')

if not WORKSPACE.exists():
    assert '<your-username>' not in REPO_URL, 'Set REPO_URL to your GitHub repo first.'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKSPACE), 'pull'], check=True)

os.chdir(WORKSPACE / 'VAR')
print(Path.cwd())


In [ ]:
!nvidia-smi
!pip install -q huggingface_hub einops matplotlib


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

weights_dir = Path('/content/VAR_weights')
weights_dir.mkdir(parents=True, exist_ok=True)

vae_path = hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=weights_dir,
)
var_path = hf_hub_download(
    repo_id='FoundationVision/var',
    filename='var_d20.pth',
    local_dir=weights_dir,
)


In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torchvision.transforms.functional as TF
from models import build_vae_var

MODEL_DEPTH = 20
patch_nums = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

vae, var = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=patch_nums,
    num_classes=1000,
    depth=MODEL_DEPTH,
    shared_aln=False,
)

vae.load_state_dict(torch.load(vae_path, map_location='cpu'), strict=True)
var.load_state_dict(torch.load(var_path, map_location='cpu'), strict=True)
vae.eval(); var.eval()
for p in vae.parameters(): p.requires_grad_(False)
for p in var.parameters(): p.requires_grad_(False)

print(f'Loaded VAR-d{MODEL_DEPTH} on {device}')


In [ ]:
DATA_ROOT = WORKSPACE
CONTENT_DIR = DATA_ROOT / 'content'
STYLE_DIR = DATA_ROOT / 'style'

content_files = sorted(CONTENT_DIR.glob('*.png'))
style_files = sorted(STYLE_DIR.rglob('*.png'))

print('content images:', len(content_files))
print('style images:', len(style_files))
print('first content:', content_files[0])
print('first style:', style_files[0])


In [ ]:
def load_var_image(path, size=256):
    img = Image.open(path).convert('RGB')
    img = ImageOps.fit(
        img,
        (size, size),
        method=Image.Resampling.LANCZOS,
        centering=(0.5, 0.5),
    )
    x = TF.to_tensor(img).mul(2).sub(1).unsqueeze(0).to(device)
    return x, img

def show_tensor_img(x, title=None):
    x = x.detach().float().cpu()
    if x.ndim == 4:
        x = x[0]
    x = x.clamp(-1, 1).add(1).div(2).permute(1, 2, 0).numpy()
    plt.imshow(x)
    if title:
        plt.title(title)
    plt.axis('off')


In [ ]:
content_path = CONTENT_DIR / '0001.png'
style_path = STYLE_DIR / 'VanGogh' / 'VanGogh001.png'

content_x, content_pil = load_var_image(content_path)
style_x, style_pil = load_var_image(style_path)

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1); plt.imshow(content_pil); plt.title('Content'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(style_pil); plt.title('Style'); plt.axis('off')
plt.show()


In [ ]:
with torch.no_grad():
    content_idx_Bl = vae.img_to_idxBl(content_x)
    style_idx_Bl = vae.img_to_idxBl(style_x)

for scale_id, (c_idx, s_idx, pn) in enumerate(zip(content_idx_Bl, style_idx_Bl, patch_nums)):
    print(
        f'scale {scale_id:02d}, patch={pn:02d}x{pn:02d}, '
        f'content={tuple(c_idx.shape)}, style={tuple(s_idx.shape)}'
    )


In [ ]:
with torch.no_grad():
    content_rec = vae.idxBl_to_img(content_idx_Bl, same_shape=True, last_one=True)
    style_rec = vae.idxBl_to_img(style_idx_Bl, same_shape=True, last_one=True)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1); show_tensor_img(content_rec, 'Content VAE reconstruction')
plt.subplot(1, 2, 2); show_tensor_img(style_rec, 'Style VAE reconstruction')
plt.show()


In [ ]:
with torch.no_grad():
    content_var_input = vae.quantize.idxBl_to_var_input(content_idx_Bl)
    style_var_input = vae.quantize.idxBl_to_var_input(style_idx_Bl)

print('content teacher-forcing input:', tuple(content_var_input.shape))
print('style teacher-forcing input:', tuple(style_var_input.shape))
print('expected token length without first scale:', sum(pn * pn for pn in patch_nums[1:]))
